# A2 — Knowledge-Base Demo

Two things this notebook shows, per `SUBMISSION.md`'s requirement for
`notebooks/kb_demo.ipynb`: **(1) OCR quality on a sample** (CER against
`grading_kit/heldout_pages/` + `labels.jsonl`, using the actual shipped OCR engine) and
**(2) one working retrieval example** against the built vector index.

## Kaggle setup (do this before "Run All")

1. Attach a dataset containing `toc.json` and the `shakespeare-ocr/` OCR-text corpus (same
   as `codes/chunk_embed_index_shakespeare.ipynb`), **or** attach a dataset with an
   already-built `index_build/` (e.g. from `codes/output_of_eci_me/` or
   `codes/output_of_eci_trjo/`) if you just want the retrieval half to work without
   rebuilding.
2. For the OCR-quality half: attach a dataset containing the 4 heldout page images named
   in `grading_kit/heldout_pages/NEEDED_IMAGES.md` (`bio_001.png`, `page_0500.png`,
   `page_0875.png`, `page_1249.png`) plus this repo's `grading_kit/labels.jsonl`. **If those
   images aren't attached yet, this notebook still runs end to end — the OCR-quality section
   just reports 0 pages scored and says why, instead of failing.** This part loads
   `Qwen/Qwen3-VL-4B-Instruct` (the shipped OCR engine, ~8GB download) and needs GPU
   (Settings -> Accelerator -> GPU T4 x2) to run in reasonable time.
3. Enable Internet (Settings -> Internet -> On) for `pip install`, the Qwen3-VL-4B-Instruct
   download, and the BGE-M3 download.


## Part 0 — Setup

In [ ]:
!pip install -q qwen-vl-utils jiwer sentence-transformers faiss-cpu pillow


In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np


## Part 1 — Locate inputs (Kaggle mount paths are unpredictable)

Same search-by-name pattern used throughout this project's other notebooks -- never
hardcode a full path.


In [ ]:
def resolve_by_name(name, search_roots=("/kaggle/input", ".", "..")):
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        if (root / name).exists():
            return root / name
        hits = list(root.rglob(name))
        if hits:
            return hits[0]
    return None

IS_KAGGLE = Path("/kaggle/input").exists()

TOC_PATH = resolve_by_name("toc.json")
LABELS_PATH = resolve_by_name("labels.jsonl")
HELDOUT_DIR = resolve_by_name("heldout_pages")
INDEX_BUILD_DIR = resolve_by_name("index_build")  # a pre-built run, if attached instead of rebuilding

print(f"[paths] toc.json      -> {TOC_PATH}")
print(f"[paths] labels.jsonl  -> {LABELS_PATH}")
print(f"[paths] heldout_pages -> {HELDOUT_DIR}")
print(f"[paths] index_build   -> {INDEX_BUILD_DIR}")


## Part 2 — OCR quality on a sample

Runs the **actual shipped OCR engine** -- `Qwen/Qwen3-VL-4B-Instruct`, prompted zero-shot
per page image, exactly as in `codes/q3_4B_gt_part1.ipynb` through `part4.ipynb` (the 4
notebooks that produced this corpus's `data/ocr_text/`) -- against each heldout page, and
scores the result against `grading_kit/labels.jsonl`. This is a genuine reproduction of the
production OCR call, not a stand-in engine: Tesseract/PaddleOCR/EasyOCR/docTR were
benchmarked and rejected for catastrophic accuracy on this corpus (`notebooks/eda.ipynb`
Stage 4), so scoring one of those here would answer the wrong question.


In [ ]:
# Same prompt used by codes/q3_4B_gt_part1..4.ipynb to build data/ocr_text/ -- kept
# identical here so this is a genuine reproduction of the production OCR call, not a
# different prompt that happens to use the same model.
QWEN_MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"
QWEN_MIN_PIXELS = 256 * 32 * 32
QWEN_MAX_PIXELS = 1024 * 32 * 32
QWEN_PROMPT = """Extract the text from this document exactly as it appears.
If the image is completely blank or contains absolutely no readable text, output EXACTLY: [Blank Page]

Otherwise, you MUST transcribe the text and adhere to these strict rules:
- NO WRAPPERS: Output ONLY the transcribed text. Do not add markdown or conversational introductions.
- ARCHAIC ENGLISH: Preserve all archaic spellings (e.g., 'thine', 'cam'st'), original punctuation, and ligatures.
- TWO-COLUMN LAYOUT: Read the left column completely from top to bottom, then the right column.
- IGNORE BLEED-THROUGH: Strictly ignore any faint or reversed text bleeding through from the back of the page."""

_BAD_PREFIXES = [
    "Here is the text", "The text", "Here's the text",
    "Extracted text:", "Text extracted:", "The text in the image is:",
]


def transcribe_with_qwen(image_path, model, processor):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(image_path),
             "min_pixels": QWEN_MIN_PIXELS, "max_pixels": QWEN_MAX_PIXELS},
            {"type": "text", "text": QWEN_PROMPT},
        ],
    }]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_prompt], images=image_inputs, videos=video_inputs,
                        padding=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=2048, use_cache=True, do_sample=False)
    trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    extracted = processor.batch_decode(trimmed, skip_special_tokens=True,
                                        clean_up_tokenization_spaces=False)[0].strip()
    for prefix in _BAD_PREFIXES:
        if extracted.lower().startswith(prefix.lower()):
            extracted = extracted.split("\n", 1)[-1].strip()
            extracted = extracted.replace("```text", "").replace("```", "").strip()
    return extracted


In [ ]:
import jiwer

results = []

if LABELS_PATH is None:
    print("[ocr-quality] labels.jsonl not found among inputs -- skipping (attach it to enable this check)")
elif HELDOUT_DIR is None:
    print("[ocr-quality] heldout_pages/ directory not found among inputs -- skipping "
          "(the 4 ground-truth labels exist in grading_kit/labels.jsonl, but the matching "
          "PNGs haven't been attached yet -- see grading_kit/heldout_pages/NEEDED_IMAGES.md)")
else:
    labels = {}
    with open(LABELS_PATH, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            labels[row["page_id"]] = row["text"]

    available = {pid: HELDOUT_DIR / f"{pid}.png" for pid in labels if (HELDOUT_DIR / f"{pid}.png").exists()}
    if not available:
        print("[ocr-quality] 0 pages scored (no matching images found under heldout_pages/)")
    else:
        import torch
        from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
        from qwen_vl_utils import process_vision_info

        print(f"[ocr-quality] loading {QWEN_MODEL_NAME} ...")
        qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
            QWEN_MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float16,
            attn_implementation="sdpa", device_map="auto",
        ).eval()
        qwen_processor = AutoProcessor.from_pretrained(
            QWEN_MODEL_NAME, trust_remote_code=True,
            min_pixels=QWEN_MIN_PIXELS, max_pixels=QWEN_MAX_PIXELS,
        )

        for page_id, image_path in available.items():
            gt_text = labels[page_id]
            t0 = time.time()
            hyp_text = transcribe_with_qwen(image_path, qwen_model, qwen_processor)
            elapsed = time.time() - t0
            cer = jiwer.cer(gt_text, hyp_text)
            wer = jiwer.wer(gt_text, hyp_text)
            results.append({"page_id": page_id, "cer": cer, "wer": wer, "time_sec": elapsed})
            print(f"[ocr-quality] {page_id}: CER={cer:.4f}  WER={wer:.4f}  ({elapsed:.1f}s)")

    if results:
        mean_cer = sum(r["cer"] for r in results) / len(results)
        mean_wer = sum(r["wer"] for r in results) / len(results)
        print(f"\n[ocr-quality] mean CER={mean_cer:.4f}  mean WER={mean_wer:.4f}  "
              f"over n={len(results)} heldout pages")


## Part 3 — One retrieval example

Loads an already-built FAISS index (either freshly built by `src/doc_agent/index/store.py`
into `data/index/`, or a pre-built `index_build/` attached as a Kaggle input) and runs one
real query -- the same canonical query used throughout `codes/A2_form.md` Section 5, so this
result is directly comparable to the two independently cross-validated full-corpus runs
already documented there.


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

def find_index_dir():
    for candidate in (Path("data/index"), INDEX_BUILD_DIR):
        if candidate and Path(candidate).exists() and (Path(candidate) / "index.faiss").exists():
            return Path(candidate)
    return None

index_dir = find_index_dir()
if index_dir is None:
    print("[retrieval] no built index found (checked data/index/ and any attached "
          "index_build/ dataset) -- run scripts/build_index.sh first, or attach a "
          "pre-built index_build/ dataset, then re-run this cell")
else:
    print(f"[retrieval] loading index from {index_dir}")
    index = faiss.read_index(str(index_dir / "index.faiss"))
    with open(index_dir / "chunk_ids.json", encoding="utf-8") as f:
        ids = json.load(f)

    # chunks_contract.jsonl has one Chunk-shaped JSON object per line, id -> text
    chunk_text_by_id = {}
    contract_path = index_dir / "chunks_contract.jsonl"
    if contract_path.exists():
        with open(contract_path, encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                chunk_text_by_id[row["id"]] = row["text"]

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer("BAAI/bge-m3", device=device)

    QUERY = "To be, or not to be, that is the question"
    qvec = model.encode([QUERY], normalize_embeddings=True, convert_to_numpy=True)
    scores, rows = index.search(qvec.astype(np.float32), 3)

    print(f"\n[retrieval example] query: {QUERY!r}")
    for score, row in zip(scores[0], rows[0]):
        if row == -1:
            continue
        cid = ids[int(row)]
        text = chunk_text_by_id.get(cid, "(text not available -- chunks_contract.jsonl missing/incomplete)")
        print(f"  score={score:.4f}  id={cid}  {text[:140]!r}...")


## Part 4 — Summary

In [ ]:
print("=" * 70)
print("KB DEMO SUMMARY")
print("=" * 70)
print(f"OCR quality  : {len(results)} heldout page(s) scored" if results else
      "OCR quality  : not run (see Part 2's message above)")
print(f"Retrieval    : {'ran (see Part 3 above)' if index_dir else 'not run (see Part 3 message above)'}")
print("=" * 70)
